# 1. Data Cleaning & Preparation
In this section, I load and prepare the data for the Fake News Detection project.

The original data comes in two separate files:

true.csv → Contains real news

fake.csv → Contains fake news

The goal of this section is to merge the files, clean up the text, and prepare the data for the next steps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

true_df = pd.read_csv("/content/Capstone/Data/true.csv")
fake_df = pd.read_csv("/content/Capstone/Data/fake.csv")

true_df["label"] = 0
fake_df["label"] = 1

df = pd.concat([true_df, fake_df], ignore_index=True)

## 1.1 Load Raw Data
I loaded the two files using pandas, then printed the data format to make sure it was loaded correctly.

In [ ]:
true_df = pd.read_csv("/content/Capstone/Data/true.csv")
fake_df = pd.read_csv("/content/Capstone/Data/fake.csv")

In [ ]:
("True shape:", true_df.shape)
print("Fake shape:", fake_df.shape)

## 1.2 Add Labels
Since each file represents a different category, we add a new column:

label = 0 → for real news

label = 1 → for fake news

This column will be used by the model later in the training process.

In [ ]:
true_df["label"] = 0   # real news
fake_df["label"] = 1   # fake news

## 1.3 Combine Both Files
After adding the categories, I combined the two files into one DataFrame so that I could work on a unified dataset.

In [ ]:
df = pd.concat([true_df, fake_df], ignore_index=True)
df.head()

## 1.4 Select Text Column
The files contain the following columns:
title, text, subject, date

However, the form only needs one text column, so use the text column as it contains the complete content of the article.

In [ ]:
df['label'] = df['label'].astype(int)
df['text'] = df['text'].astype(str)

## 1.5 Handling Missing Values
We do the following:

Delete rows that contain no text.

Fill any missing values ​​with an empty string "".

Ensure all text is of type string.

This prevents errors during training.

In [ ]:
if "text" in df.columns:
    df["text"] = df["text"].astype(str)
elif "content" in df.columns:
    df["text"] = df["content"].astype(str)
elif "body" in df.columns:
    df["text"] = df["body"].astype(str)
else:
    raise ValueError("No text column found in dataset!")

## 1.6 Remove Duplicates
The presence of duplicate articles may cause form bias, so delete any duplicate rows.

In [ ]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].fillna("")

## 1.7 Text Cleaning
I clean the text using the following steps:

Convert the text to lowercase

Remove punctuation and symbols

Remove anything other than English letters

Remove stopwords

Regroup the words into clean text

The output is stored in a new column called clean_text.
This is the column that will be used in TF-IDF and LSTM.

In [ ]:
df = df.drop_duplicates()

## 1.8 Save Cleaned Dataset
After completing the cleanup, I save the final file as:

Writing Code
cleaned_fake_news.csv
This will be used in:

Step 02 → Machine Learning Models

Step 03 → Neural Network Model (LSTM)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords") # Ensure stopwords are downloaded
stop_words = set(stopwords.words("english")) # Define stop_words

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()
df.to_csv("cleaned_fake_news.csv", index=False)
print("Saved cleaned dataset!")